In [ ]:
import os
import sys
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from pathlib import Path

# Walk up from cwd until src/mt2d_inv is found (works under tests/*)
_project_root = None
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "src" / "mt2d_inv").is_dir():
        _project_root = str(_p)
        break
if _project_root is None:
    raise RuntimeError("Could not find MTinv_OT project root (src/mt2d_inv)")
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

import torch
import numpy as np
import matplotlib.pyplot as plt
from src.mt2d_inv import MT2DInverterWeightedCost
from src.mt2d_inv.io import ExperimentLogger

logger = ExperimentLogger(
    model_tag="Cascadia",
    output_root=Path("test_results"),
)

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")


In [ ]:
PLOT_CONFIG = {
    "cmap": "jet_r",
    "clip_to_stations": True,
    "ylim": [200, 0],
    "profile_extend_km": 20,
    "vmin": 0,
    "vmax": 4,
    "mc_cmap": "jet_r",
    "mc_clip_to_stations": True,
    "mc_ylim": [200, 0],
    "mc_profile_extend_km": 20,
    "mc_vmin": 0,
    "mc_vmax": 4,
    "im_cmap": "jet_r",
    "im_clip_to_stations": True,
    "im_ylim": [200, 0],
    "im_profile_axis_width_km": 500,
    "df_station_indices": "all",
    "df_plot_noise_cap": None,
    "pf_depth_limit_km": 50,
}

save_plot_kwargs = PLOT_CONFIG


In [ ]:
from __future__ import annotations

import torch
from src.mt2d_inv.data_prep import PrepareData

EDI_DIR = "Cascadia-CAFE-profile-3"
FREQ_MIN_HZ = 1e-5
FREQ_MAX_HZ = 1e-1
MAG_DECLINATION_DEG = 15.0
STRIKE_TRUE_DEG = 0.0
STRIKE_MAGNETIC_DEG = float(STRIKE_TRUE_DEG - MAG_DECLINATION_DEG)

prep = PrepareData(
    edi_dir=EDI_DIR,
    mag_declination_deg=MAG_DECLINATION_DEG,
    edi_impedance_unit="mv/km/nt",
    freq_min_hz=FREQ_MIN_HZ,
    freq_max_hz=FREQ_MAX_HZ,
    clean_data=False,
)

device = "cuda" if torch.cuda.is_available() else "cpu"

freqs_t, stations_t, data_dict = prep.run_all_simple(
    rotate=True,
    strike_true_deg=STRIKE_TRUE_DEG,
    strike_magnetic_deg=STRIKE_MAGNETIC_DEG,
    device=device,
)
print(f"[pipeline] Fixed strike (true N) = {STRIKE_TRUE_DEG}°, rotation (mag) = {STRIKE_MAGNETIC_DEG:.2f}°")
print(f"[pipeline] Stations: {len(prep.mt_objects)} (all EDI kept; cleaning only masks bad points)")


In [ ]:
nza = 10
z_air = -np.logspace(np.log10(10), np.log10(50000), nza)
z_air = np.flip(z_air)
z_air = np.append(z_air, 0)

nz = 80
z_sub = np.logspace(np.log10(10), np.log10(300000), nz)
zn = np.concatenate([z_air[:-1], np.array([0]), z_sub])

y_center = np.linspace(-250000, 250000, 201)
y_left = -np.logspace(np.log10(255000), np.log10(950000), 20)
y_right = np.logspace(np.log10(255000), np.log10(950000), 20)
y_left = np.flip(y_left)
yn = np.concatenate([y_left, y_center, y_right])

print("新的网格统计:")
print(f" - 垂向第一层厚度: {zn[nza+1] - zn[nza]:.1f} m")
print(f" - 最大深度: {zn[-1]/1000:.1f} km")
print(f" - 中心区横向分辨率: {y_center[1] - y_center[0]:.1f} m")
print(f" - 网格总数: {len(zn)-1} x {len(yn)-1}")


In [ ]:
inv = MT2DInverterWeightedCost(
    yn=torch.as_tensor(yn, dtype=torch.float64, device=device),
    zn=torch.as_tensor(zn, dtype=torch.float64, device=device),
    nza=nza,
    freqs=freqs_t,
    stations=stations_t,
    device=device,
    te_weight=3.0,
    tm_weight=1.0,
    data_loss_scale=50,
)
inv.load_obs_data(data_dict, noise_floor=0.05)
w_d_point = inv.update_ot_w_d_per_point_from_noise(
    w_d_scale=[1.5, 1, 1, 1],
    normalize="mean",
)


In [ ]:
inv.set_forward_operator()
inv.initialize_model(initial_sigma=0.03)

inv.plot_initial_model(
    clip_to_stations=PLOT_CONFIG["im_clip_to_stations"],
    ylim=PLOT_CONFIG["im_ylim"],
    profile_axis_width_km=PLOT_CONFIG["im_profile_axis_width_km"],
)

final_sigma = inv.run_inversion(
    n_epochs=200,
    mode="mse",
    progress_interval=10,
    current_lambda=0.1,
    use_adaptive_lambda=True,
    lr=0.8,
    update_interval=10,
    norm_type="L2",
    alpha=0.8,
    rms_chi2_stop=1.05,
    use_depth_weights=True,
    use_ot_weights=True,
    enable_blur_anneal=True,
    warmup_epochs=0,
)

print("done; sigma range:", final_sigma.min().item(), final_sigma.max().item())


In [ ]:
inv.plot_loss_history()

inv.plot_model_comparison(
    cmap=PLOT_CONFIG["mc_cmap"],
    clip_to_stations=PLOT_CONFIG["mc_clip_to_stations"],
    ylim=PLOT_CONFIG["mc_ylim"],
    profile_extend_km=PLOT_CONFIG["mc_profile_extend_km"],
    vmin=PLOT_CONFIG["mc_vmin"],
    vmax=PLOT_CONFIG["mc_vmax"],
)

print(freqs_t.min().item(), freqs_t.max().item())
batch_size = 3
n_stations = len(inv.stations)
for start in range(0, n_stations, batch_size):
    end = min(start + batch_size, n_stations)
    station_indices = list(range(start, end))
    inv.plot_data_fitting(station_indices=station_indices)
inv.plot_gradient_history()


In [ ]:
inv.plot_roughness_misfit_curve()


In [ ]:
logger.save_from_inverter(
    inv,
    run_name="mse_epoch200",
    plot_kwargs=save_plot_kwargs,
)
